<a href="https://colab.research.google.com/github/nastyaland/Stochastic-MaGNet/blob/Data-collection-and-Cleaning/data_collection/Nasdaq100_data_collection_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import yfinance as yf
import pandas as pd
import requests
import io

print("Fetching NASDAQ 100 component list...")
url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    tables = pd.read_html(io.StringIO(response.text))
    for table in tables:
        if 'Ticker' in table.columns:
            tickers = table['Ticker'].tolist()
            break
except Exception as e:
    print(f"Error fetching from Wikipedia: {e}")

tickers = [ticker.replace('.', '-') for ticker in tickers if ticker]

# yfinance's end date is exclusive
start_date = '2020-01-01'
end_date = '2025-01-01'

print(f"\nDownloading raw data for {len(tickers)} tickers...")
# auto_adjust=False ensures we obtain the underlying actual transaction price
raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)

# Core Step: Converting data from a wide table to standard panel data format
# This transforms features (Open, High...) into columns while converting (Date, Ticker) into row indices
print("\nRestructuring data into Quant Panel format...")
df_panel = raw_data.stack(level=1, future_stack=True).swaplevel().sort_index()
df_panel.index.names = ['Ticker', 'Date']

# Retain only the 5 core fundamental features mentioned in the paper
base_features = ['Open', 'High', 'Low', 'Close', 'Volume']
df_features = df_panel[base_features]

print("\n--- Data Ready for Feature Engineering ---")
print(f"DataFrame Shape: {df_features.shape}")
print(f"Contains NaNs? {df_features.isnull().values.any()}")

# Print the header data to give you a visual sense of the structure
print("\nPeek at the formatted DataFrame:")
print(df_features.head(10))

Fetching NASDAQ 100 component list...



[*********************100%***********************]  101 of 101 completed



Restructuring data into Quant Panel format...

--- Data Ready for Feature Engineering ---
DataFrame Shape: (127058, 5)
Contains NaNs? True

Peek at the formatted DataFrame:
Price                   Open       High        Low      Close       Volume
Ticker Date                                                               
AAPL   2020-01-02  74.059998  75.150002  73.797501  75.087502  135480400.0
       2020-01-03  74.287498  75.144997  74.125000  74.357498  146322800.0
       2020-01-06  73.447502  74.989998  73.187500  74.949997  118387200.0
       2020-01-07  74.959999  75.224998  74.370003  74.597504  108872000.0
       2020-01-08  74.290001  76.110001  74.290001  75.797501  132079200.0
       2020-01-09  76.809998  77.607498  76.550003  77.407501  170108400.0
       2020-01-10  77.650002  78.167503  77.062500  77.582497  140644800.0
       2020-01-13  77.910004  79.267502  77.787498  79.239998  121532000.0
       2020-01-14  79.175003  79.392502  78.042503  78.169998  161954400.0
 

In [3]:
# This block is for mandatory cleaning the files gnerated in the directory, you shouldn't touch it if it's not necessary

# Force remove the CSV directory and all its contents
!rm -rf ./qlib_csv_data

# Force remove the binary directory if it exists
!rm -rf ./qlib_bin_data

print("Dirty directories successfully nuked! Ready for clean data.")

Dirty directories successfully nuked! Ready for clean data.


In [4]:
print("--Data Quality Check--")
valid_days_count = df_features['Close'].groupby('Ticker').count()

max_trading_days = valid_days_count.max()
print(f"Max Trading Days: {max_trading_days}\n")

#Completelt empty ticker
empty_tickers = valid_days_count[valid_days_count == 0].index.tolist()
print(f"Completely empty Tickers ({len(empty_tickers)}): {empty_tickers}\n")

#Ticker with partial missing data
incomplete_tickers = valid_days_count[(valid_days_count >0) & (valid_days_count < max_trading_days)]
print(f"\nTickers with incomplete data ({len(incomplete_tickers)}):")
for ticker,count in incomplete_tickers.items():
    missing = max_trading_days - count
    print(f" - {ticker}: Missing {missing} days (Has {count} valid days)")

clean_tickers = valid_days_count[valid_days_count == max_trading_days].index.tolist()
df_clean = df_features.loc[clean_tickers]
print("\nStripping timezone from df_clean dates...")
df_clean = df_clean.reset_index()
df_clean['Date'] = df_clean['Date'].dt.tz_localize(None)
df_clean = df_clean.set_index(['Ticker','Date'])
print("\n--- Filtering Complete ---")
print(f"Original number of tickers: {len(valid_days_count)}")
print(f"Clean number of tickers retained: {len(clean_tickers)}")
print(f"Clean DataFrame Shape: {df_clean.shape}")
print(f"Contains NaNs now? {df_clean.isnull().values.any()}")

--Data Quality Check--
Max Trading Days: 1258

Completely empty Tickers (0): []


Tickers with incomplete data (7):
 - ABNB: Missing 238 days (Has 1020 valid days)
 - APP: Missing 323 days (Has 935 valid days)
 - ARM: Missing 931 days (Has 327 valid days)
 - CEG: Missing 516 days (Has 742 valid days)
 - DASH: Missing 237 days (Has 1021 valid days)
 - GEHC: Missing 745 days (Has 513 valid days)
 - PLTR: Missing 188 days (Has 1070 valid days)

Stripping timezone from df_clean dates...

--- Filtering Complete ---
Original number of tickers: 101
Clean number of tickers retained: 94
Clean DataFrame Shape: (118252, 5)
Contains NaNs now? False


In [5]:
print(df_clean.head())
print(df_clean.tail())

Price                   Open       High        Low      Close       Volume
Ticker Date                                                               
AAPL   2020-01-02  74.059998  75.150002  73.797501  75.087502  135480400.0
       2020-01-03  74.287498  75.144997  74.125000  74.357498  146322800.0
       2020-01-06  73.447502  74.989998  73.187500  74.949997  118387200.0
       2020-01-07  74.959999  75.224998  74.370003  74.597504  108872000.0
       2020-01-08  74.290001  76.110001  74.290001  75.797501  132079200.0
Price                    Open        High         Low       Close     Volume
Ticker Date                                                                 
ZS     2024-12-24  185.979996  187.589996  184.679993  187.259995   654200.0
       2024-12-26  186.000000  188.500000  185.869995  187.630005   806500.0
       2024-12-27  185.259995  185.929993  181.259995  184.559998  1256100.0
       2024-12-30  181.529999  184.404999  180.100006  183.130005  1032400.0
       2024-1

In [6]:
!pip install pyqlib
import os
import pandas as pd

print("Preparing CSV files for Qlib...")

csv_dir = './qlib_csv_data'
os.makedirs(csv_dir, exist_ok = True)
df_qlib = df_clean.copy()
df_qlib.columns = [col.lower() for col in df_qlib.columns]
df_qlib = df_qlib.reset_index()
for ticker, group in df_qlib.groupby('Ticker'):
  file_path = os.path.join(csv_dir, f"{ticker}.csv")
  group.to_csv(file_path, index=False, columns=['Date','open','high','low','close','volume'])
print(f"Saved CSVs for {len(df_qlib['Ticker'].unique())} tickers in '{csv_dir}'.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147

In [7]:
# 1. Create the binary directory
!mkdir -p ./qlib_bin_data

# 2. Download the official conversion script directly from Microsoft's GitHub
!wget -qO dump_bin.py https://raw.githubusercontent.com/microsoft/qlib/main/scripts/dump_bin.py

# 3. Execute the script directly (Updated parameter name)
!python dump_bin.py dump_all --data_path ./qlib_csv_data --qlib_dir ./qlib_bin_data --date_field_name Date --symbol_field_name Ticker

2026-03-11 20:46:33.005 | INFO     | __main__:_get_all_date:307 - start get all date......
100% 94/94 [00:02<00:00, 33.72it/s]
2026-03-11 20:46:35.799 | INFO     | __main__:_get_all_date:326 - end of get all date.

2026-03-11 20:46:35.800 | INFO     | __main__:_dump_calendars:329 - start dump calendars......
2026-03-11 20:46:35.858 | INFO     | __main__:_dump_calendars:332 - end of calendars dump.

2026-03-11 20:46:35.858 | INFO     | __main__:_dump_instruments:335 - start dump instruments......
2026-03-11 20:46:35.870 | INFO     | __main__:_dump_instruments:337 - end of instruments dump.

2026-03-11 20:46:35.870 | INFO     | __main__:_dump_features:340 - start dump features......
100% 94/94 [00:03<00:00, 29.00it/s]
2026-03-11 20:46:39.112 | INFO     | __main__:_dump_features:347 - end of features dump.



In [8]:
import qlib
from qlib.config import REG_US
from qlib.contrib.data.handler import Alpha158, Alpha360
import pandas as pd

print ("Initializing Qlib environment...")
qlib.init(provider_uri = './qlib_bin_data', region = REG_US)
start_time = "2020-01-01"
end_time = "2024-12-31"

print("\nCalculating Alpha158 factors...")
handler_158 = Alpha158(instruments="all",start_time = start_time, end_time = end_time)
df_alpha158 = handler_158.fetch()

print(f"Raw shape of df_alpha158: {df_alpha158.shape}")
# Correct approach for data cleaning: First, use 'dropna(axis=1, how='all')' to remove columns like 'VWAP0' that are 100% corrupted.
# Then, use dropna() to remove the initial 60-day warm-up period, and finally take the first 5 rows.
print("\n[Real Sanity Check] Here are the actual calculated numbers:")
print(df_alpha158.dropna(axis=1, how='all').dropna().head())

print("Calculating Alpha360 factors...")
handler_360 = Alpha360(instruments="all",start_time = start_time, end_time = end_time, fit_start_time=start_time, fit_end_time=end_time)
df_alpha360 = handler_360.fetch()

print("\n-- Merging Features --")
df_clean.columns = [str(col).lower() for col in df_clean.columns]

df_clean_matched = df_clean.swaplevel().sort_index()
df_clean_matched.index.names = ['datetime', 'instrument']

#Concatenate the base features with Alpha factor sets
df_combined = pd.concat([df_clean_matched, df_alpha158, df_alpha360], axis=1)
print(f"Combined DataFrame Shape: {df_combined.shape}")

df_combined.head()

[645:MainThread](2026-03-11 20:46:42,750) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


Initializing Qlib environment...


[645:MainThread](2026-03-11 20:46:45,491) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[645:MainThread](2026-03-11 20:46:45,493) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/content/qlib_bin_data')}



Calculating Alpha158 factors...


[645:MainThread](2026-03-11 20:47:01,462) INFO - qlib.timer - [log.py:127] - Time cost: 15.967s | Loading data Done
[645:MainThread](2026-03-11 20:47:01,591) INFO - qlib.timer - [log.py:127] - Time cost: 0.039s | DropnaLabel Done
[645:MainThread](2026-03-11 20:47:03,516) INFO - qlib.timer - [log.py:127] - Time cost: 1.924s | CSZScoreNorm Done
[645:MainThread](2026-03-11 20:47:03,520) INFO - qlib.timer - [log.py:127] - Time cost: 2.056s | fit & process data Done
[645:MainThread](2026-03-11 20:47:03,524) INFO - qlib.timer - [log.py:127] - Time cost: 18.029s | Init data Done


Raw shape of df_alpha158: (118252, 159)

[Real Sanity Check] Here are the actual calculated numbers:
                           KMID      KLEN     KMID2       KUP      KUP2  \
datetime   instrument                                                     
2020-03-30 AAPL        0.016232  0.024408  0.665030  0.002832  0.116014   
           ADBE        0.030755  0.048431  0.635027  0.014827  0.306149   
           ADI         0.015971  0.043221  0.369509  0.005584  0.129199   
           ADP         0.040937  0.051133  0.800594  0.005438  0.106350   
           ADSK        0.092266  0.111321  0.828823  0.008897  0.079924   

                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
datetime   instrument                                                    ...   
2020-03-30 AAPL        0.005344  0.218956  0.018745  0.767972  0.984027  ...   
           ADBE        0.002849  0.058824  0.018777  0.387702  0.970162  ...   
           ADI         0.021666  0.501292  0.032053  

[645:MainThread](2026-03-11 20:47:23,777) INFO - qlib.timer - [log.py:127] - Time cost: 20.019s | Loading data Done
[645:MainThread](2026-03-11 20:55:34,301) INFO - qlib.timer - [log.py:127] - Time cost: 490.323s | ProcessInf Done
/usr/local/lib/python3.12/dist-packages/qlib/data/dataset/processor.py:241: RuntimeWarning: Mean of empty slice
  self.mean_train = np.nanmean(df[cols].values, axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
[645:MainThread](2026-03-11 20:55:35,599) INFO - qlib.timer - [log.py:127] - Time cost: 1.294s | ZScoreNorm Done
[645:MainThread](2026-03-11 20:55:36,252) INFO - qlib.timer - [log.py:127] - Time cost: 0.650s | Fillna Done
[645:MainThread](2026-03-11 20:55:36,395) INFO - qlib.timer - [log.py:127] - Time cost: 0.070s | DropnaLabel Done
[645:MainThread](2026-03-11 20:55:38,063) INFO - qlib.timer - [log.py:127]


-- Merging Features --
Combined DataFrame Shape: (118252, 525)


open        high         low       close  \
datetime   instrument                                                   
2020-01-02 AAPL         74.059998   75.150002   73.797501   75.087502   
           ADBE        330.000000  334.480011  329.170013  334.429993   
           ADI         120.110001  120.650002  118.739998  120.430000   
           ADP         171.960007  172.000000  169.199997  170.320007   
           ADSK        184.210007  187.889999  181.880005  187.830002   

                            volume      KMID      KLEN     KMID2       KUP  \
datetime   instrument                                                        
2020-01-02 AAPL        135480400.0  0.013874  0.018262  0.759707  0.000844   
           ADBE          1990100.0  0.013424  0.016091  0.834274  0.000152   
           ADI           1279300.0  0.002664  0.015902  0.167539  0.001832   
           ADP           1364600.0 -0.009537  0.016283 -0.585713  0.000233   
           ADSK          1515000.0  0.019651  0.032626  0.602329  0.000326   

                           KUP2  ...  VOLUME8  VOLUME7  VOLUME6  VOLUME5  \
datetime   instrument            ...                                       
2020-01-02 AAPL        0.046211  ...      0.0      0.0      0.0      0.0   
           ADBE        0.009420  ...      0.0      0.0      0.0      0.0   
           ADI         0.115184  ...      0.0      0.0      0.0      0.0   
           ADP         0.014283  ...      0.0      0.0      0.0      0.0   
           ADSK        0.009983  ...      0.0      0.0      0.0      0.0   

                       VOLUME4  VOLUME3  VOLUME2  VOLUME1   VOLUME0    LABEL0  
datetime   instrument                                                          
2020-01-02 AAPL            0.0      0.0      0.0      0.0  0.085742  0.270136  
           ADBE            0.0      0.0      0.0      0.0  0.085742  0.184407  
           ADI             0.0      0.0      0.0      0.0  0.085742 -0.483767  
           ADP             0.0      0.0      0.0      0.0  0.085742  0.017206  
           ADSK            0.0      0.0      0.0      0.0  0.085742  0.414086  

[5 rows x 525 columns]

In [9]:
print(df_alpha158.head())
print(df_alpha360.tail())

                           KMID      KLEN     KMID2       KUP      KUP2  \
datetime   instrument                                                     
2020-01-02 AAPL        0.013874  0.018262  0.759707  0.000844  0.046211   
           ADBE        0.013424  0.016091  0.834274  0.000152  0.009420   
           ADI         0.002664  0.015902  0.167539  0.001832  0.115184   
           ADP        -0.009537  0.016283 -0.585713  0.000233  0.014283   
           ADSK        0.019651  0.032626  0.602329  0.000326  0.009983   

                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
datetime   instrument                                                    ...   
2020-01-02 AAPL        0.003544  0.194083  0.016574  0.907579  0.986316  ...   
           ADBE        0.002515  0.156306  0.015788  0.981161  0.986754  ...   
           ADI         0.011406  0.717278  0.012239  0.769633  0.997343  ...   
           ADP         0.006513  0.400003 -0.003256 -0.199993  1.009629  .

In [10]:
rows_with_nan = df_combined.isnull().any(axis=1).sum()
total_rows = len(df_combined)
print(f"Total rows before cleaning: {total_rows}")
print(f"Rows containing at least one NaN: {rows_with_nan}")
print(f"Percentage of rows to be dropped: {(rows_with_nan/total_rows)*100:.2f}%\n")

print("Top 20 features with the most NaNs:")
nan_counts = df_combined.isnull().sum().sort_values(ascending=False)
print(nan_counts.head(20))
print("\nPeek at the rows with NaNs (Notice the dates):")
rows_with_nan_df = df_combined[df_combined.isnull().any(axis=1)]
print(rows_with_nan_df.head(10))

Total rows before cleaning: 118252
Rows containing at least one NaN: 118252
Percentage of rows to be dropped: 100.00%

Top 20 features with the most NaNs:
VWAP0     118252
ROC60       5640
ROC30       2820
ROC20       1880
CORD5       1149
CORD10      1033
ROC10        940
CORD20       852
CORD30       732
CORR5        692
RSQR5        691
CORD60       525
ROC5         470
CORR10       416
RSQR10       415
WVMA60       188
WVMA30       188
WVMA5        188
WVMA10       188
LABEL0       188
dtype: int64

Peek at the rows with NaNs (Notice the dates):
                             open        high         low       close  \
datetime   instrument                                                   
2020-01-02 AAPL         74.059998   75.150002   73.797501   75.087502   
           ADBE        330.000000  334.480011  329.170013  334.429993   
           ADI         120.110001  120.650002  118.739998  120.430000   
           ADP         171.960007  172.000000  169.199997  170.320007   
      

In [11]:
threshold = len(df_combined) * 0.9
df_filtered_cols = df_combined.dropna(axis=1, thresh = threshold)

df_final = df_filtered_cols.dropna()
print(f"Final Cleaned Shape: {df_final.shape}")
print(f"Total Stocks: {len(df_final.index.get_level_values('instrument').unique())}")
print(f"Total Trading Days: {len(df_final.index.get_level_values('datetime').unique())}")

Final Cleaned Shape: (111507, 524)
Total Stocks: 94
Total Trading Days: 1196


In [12]:
days_per_stock = df_final.groupby('instrument').size()
target_days = days_per_stock.max()
good_stocks = days_per_stock[days_per_stock == target_days].index.tolist()
bad_stocks = days_per_stock[days_per_stock < target_days].index.tolist()

print(f"Standard trading days: {target_days}")
print(f"Perfectly matched stocks: {len(good_stocks)}")
print(f"Stocks need to be removed: {bad_stocks}")
df_perfect = df_final.loc[slice(None), good_stocks, :]

print(df_perfect.head())
print(df_perfect.tail())

Standard trading days: 1196
Perfectly matched stocks: 93
Stocks need to be removed: ['FER']
                             open        high         low       close  \
datetime   instrument                                                   
2020-03-30 AAPL         62.685001   63.880001   62.349998   63.702499   
           ADBE        308.890015  322.970001  308.010010  318.390015   
           ADI          89.540001   91.470001   87.599998   90.970001   
           ADP         132.399994  138.539993  131.770004  137.820007   
           ADSK        142.740005  157.179993  141.289993  155.910004   

                            volume      KMID      KLEN     KMID2       KUP  \
datetime   instrument                                                        
2020-03-30 AAPL        167976400.0  0.016232  0.024408  0.665030  0.002832   
           ADBE          4340200.0  0.030755  0.048431  0.635027  0.014827   
           ADI           3268200.0  0.015971  0.043221  0.369509  0.005584   
      

In [13]:
print(df_perfect.shape)

(111228, 524)


In [14]:
# From Dataset.py the dimension of the tensor should be (N * T * F) (Stock, Timesteps. Features)
import torch
import numpy as np

cols = df_perfect.columns.tolist()
if 'close' in cols:
  cols.insert(0, cols.pop(cols.index('close')))
df_ready = df_perfect[cols]

df_ready = df_ready.sort_index(level = ['instrument', 'datetime'])

n_stocks = 93
n_features = 532
days_per_stock = df_ready.groupby('instrument').size()
min_days = days_per_stock.min()
print(f"Minimum trading days across all stocks: {min_days}")
df_cube = df_ready.groupby('instrument').tail(min_days)
df_cube.head()

Minimum trading days across all stocks: 1196


,,close,open,high,low,volume,KMID,KLEN,KMID2,KUP,KUP2,...,VOLUME7,VOLUME6,VOLUME5,VOLUME4,VOLUME3,VOLUME2,VOLUME1,VOLUME0,LABEL0,LABEL0
datetime,instrument,,,,,,,,,,,,,,,,,,,,,
2020-03-30,AAPL,63.702499,62.685001,63.880001,62.349998,167976400.0,0.016232,0.024408,0.665030,0.002832,0.116014,...,-0.008854,-0.009429,-0.008477,-0.012348,-0.010111,-0.009694,-0.008475,0.085742,-0.052617,-2.046417
2020-03-31,AAPL,63.572498,63.900002,65.622498,63.000000,197002000.0,-0.005125,0.041041,-0.124882,0.026956,0.656815,...,-0.008854,-0.009429,-0.008477,-0.012348,-0.010111,-0.009694,-0.008475,0.085742,0.016687,0.603499
2020-04-01,AAPL,60.227501,61.625000,62.180000,59.782501,176218400.0,-0.022677,0.038905,-0.582899,0.009006,0.231491,...,-0.008854,-0.009429,-0.008477,-0.012348,-0.010111,-0.009694,-0.008475,0.085742,-0.014371,-0.584046
2020-04-02,AAPL,61.232498,60.084999,61.287498,59.224998,165934000.0,0.019098,0.034326,0.556363,0.000915,0.026667,...,-0.008854,-0.009429,-0.008477,-0.012348,-0.010111,-0.009694,-0.008475,0.085742,0.087237,3.301096
2020-04-03,AAPL,60.352501,60.700001,61.424999,59.742500,129880000.0,-0.005725,0.027718,-0.206538,0.011944,0.430906,...,-0.008854,-0.009429,-0.008477,-0.012348,-0.010111,-0.009694,-0.008475,0.085742,-0.011582,-0.477402


In [15]:
data_3d = df_cube.values.reshape(n_stocks, min_days, n_features)
data_tensor = torch.from_numpy(data_3d).float()
print(f"Formatted Data Shape: {data_tensor.shape}")
torch.save(data_tensor, 'my_nas100_2025_data.pt')

Formatted Data Shape: torch.Size([93, 1196, 532])


In [1]:
# Upload all MaGNet model training files and run the command below to train

!python train.py

python3: can't open file '/content/train.py': [Errno 2] No such file or directory
